# Dynamic RFQ market making and position execution

This notebook extends the static edge-consistent RFQ responder into a
**finite-horizon dynamic controller** that simultaneously:

1. responds to incoming RFQs with a selection-adjusted quote (or declines), and
2. executes actively in the market when RFQ flow alone cannot manage inventory.

Everything runs on synthetic data with fixed seeds, and every model, simulator,
solver, and plot lives in the `rfq_edge` package; this notebook only imports,
calls, displays, and interprets.

**Method statement (read this once).** The dynamic layer solves a
finite-state, finite-horizon Bellman problem by exact backward induction on a
grid of (time step, integer inventory, market regime). It is a **discrete
Bellman approximation to a jump-HJB**, not an exact continuous-time PDE
solution. The Bellman residual and its tolerance are reported in Section 10.

In [ ]:
import dataclasses

import numpy as np
import pandas as pd

from rfq_edge import control_plots, plots
from rfq_edge.bellman import bellman_residual, fill_continuation_delta, solve_bellman
from rfq_edge.config import OptimizerConfig
from rfq_edge.control_config import (
    MarketRegime,
    acquisition_episode,
    liquidation_episode,
    market_making_episode,
)
from rfq_edge.control_evaluation import (
    evaluate_control_policies,
    run_ablation_study,
    run_control_sensitivity,
    summarize_episode_log,
)
from rfq_edge.control_pipeline import (
    POLICY_ORDER,
    build_control_artifacts,
    make_policies,
    solve_episode_policies,
)
from rfq_edge.control_reporting import (
    format_paired_differences,
    policy_comparison_table,
    reconcile_episode_rewards,
    reward_decomposition_table,
    rfq_decision_across_inventories,
    rfq_increment_curves,
    training_history_diagnostics,
)
from rfq_edge.control_state import ControlState, RFQEvent
from rfq_edge.event_simulator import simulate_episode
from rfq_edge.fill_model import counterfactual_fill_curve, evaluate_fill_model
from rfq_edge.market_dynamics import simulate_exogenous_path
from rfq_edge.pipeline import cold_start_comparison, fit_framework, score_rfq
from rfq_edge.responders import observable_view
from rfq_edge.selection_model import (
    evaluate_selection_model,
    format_selection_metrics,
    make_selection_target,
    predict_selection,
)
from rfq_edge.simulation_diagnostics import (
    append_oracle_objective,
    build_oracle_context,
    oracle_optimal_quote,
    oracle_selection,
    win_rate_by_aggressiveness_bucket,
)
from rfq_edge.synthetic import SyntheticConfig, make_synthetic_rfqs
from rfq_edge.value_model import evaluate_value_models

SEED = 42
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Part I — The economic problem

## Section 1 — What is being controlled?

The dealer holds one instrument, an inventory `I` in normalized units
(one unit = $100,000 notional), and faces a finite horizon of `T` decision
steps. At every step it has **two controls**:

* **`q`** — the price quoted on an incoming RFQ (or a decline). Quoting
  controls *both* the probability of trading *and* the expected value of the
  bond conditional on trading, because clients trade on information.
* **`u`** — active execution performed outside the RFQ channel, on a discrete
  grid `u in {-2, -1, 0, +1, +2}` units, paying half-spread, temporary impact,
  and a ticket fee.

Three economic activities emerge from the *same* optimization — they are never
hard-coded:

* **market making** — quoting an RFQ for standalone selection-adjusted edge;
* **passive execution** — quoting an RFQ *because its fill moves inventory
  toward the target*, even at negative standalone edge;
* **active execution** — paying spread and impact to move inventory when RFQ
  flow is insufficient or the deadline is near.

In [ ]:
fig, ax = control_plots.plot_control_architecture()

One state feeds two controls. RFQ responses earn (or lose) the t+5 conditional
clean edge; active execution only ever costs money. Both change inventory, and
inventory carries running and terminal penalties. The controller trades these
off through one value function.

## Section 2 — Define the state

The control state is

$$S_t = (X_t,\; I_t,\; I^*_t,\; T - t,\; R_t)$$

* `X_t` — the observable market and RFQ state: CP+, volatility, market width,
  liquidity, and the current RFQ's side, size, and client tier (if one arrived);
* `I_t` — current inventory in units;
* `I^*_t` — target inventory (0 for a pure market maker);
* `T - t` — steps remaining before the horizon;
* `R_t` — the observable market regime (`CALM_LIQUID`, `NORMAL`,
  `STRESSED_ILLIQUID`), which drives arrivals, widths, volatility, execution
  costs, and the strength of client information.

A concrete example: a dealer long 8 units that must be flat in 20 steps.

In [ ]:
example_state = ControlState(
    time_index=20,
    time_remaining=20,
    market_regime=MarketRegime.NORMAL,
    inventory=8,
    target_inventory=0,
    initial_inventory=8,
    inventory_limit=12,
    current_cp_plus=100.0,
    volatility=0.08,
    liquidity_score=0.55,
    market_width=0.12,
    active_execution_available=True,
    current_rfq=None,
)
pd.Series(dataclasses.asdict(example_state))

Nothing in the state says "you are an execution desk": the target, the clock,
and the penalties are the only difference between market making and execution.

# Part II — The simulated environment

## Section 3 — Simulate the market

The event-driven market has a three-state Markov regime process, a CP+ price
path whose volatility depends on the regime, Bernoulli RFQ arrivals whose
intensity, size, and width depend on the regime, and — for each RFQ — a
**hidden client signal `h`** that is correlated with the bond's future value
and shifts the client's willingness to trade. Hidden variables exist only
inside the simulator; fitted models never see them.

In [ ]:
artifacts = build_control_artifacts(random_state=0)
mm_config = market_making_episode()
demo_path = simulate_exogenous_path(artifacts.market_config, mm_config, random_state=10)
fig, axes = control_plots.plot_market_regime_path(
    demo_path, title="One simulated episode: regimes, CP+ path, and RFQ arrivals"
)

Green/yellow/red shading marks calm, normal, and stressed regimes. RFQ
arrivals thin out and the CP+ path roughens when the market is stressed. The
bottom panel shows each RFQ's hidden client signal `h` — the up/down triangles
are client-sell/client-buy requests. This panel is labelled *simulation only*:
it is the ground truth the oracle uses and the fitted policies never observe.

The control models are fitted on a separate observable history of 20,000
historical RFQs quoted under a randomized legacy rule (so the fill model sees
support across the whole aggressiveness grid).

In [ ]:
history = artifacts.training_history
print(f"Training history: {len(history):,} RFQs")
print()
print("Regime mix:")
print(history["regime"].value_counts(normalize=True).round(3).to_string())
print()
print(f"Overall win rate: {history['won'].mean():.3f}")
print(f"Observable columns only: {list(history.columns)}")

## Section 4 — Demonstrate adverse selection

Before fitting anything, the raw history shows the mechanism the controller
must respect: **more aggressive quotes win more often**, and **winning is
systematically costly** because clients trade when their signal says the
dealer's price is wrong.

In [ ]:
diagnostics = training_history_diagnostics(history)
print("Win rate by quoted aggressiveness (z > 0 is more aggressive):")
print(diagnostics["win_rate_by_aggressiveness"].round(3).to_string(index=False))
print()
print("Realized selection D = side_sign x (V0 - y5) on fills, by dealer side:")
print(diagnostics["realized_selection_by_side"].round(3).to_string(index=False))

Both dealer sides lose about 4-5 cents per unit *on average* to the client's
information: dealer-buy wins select bonds about to go down, dealer-sell wins
select bonds about to go up. The oracle can compute the exact truth by
integrating over the hidden signal:

In [ ]:
z_grid = np.asarray(artifacts.market_config.aggressiveness_grid)
fig, axes = control_plots.plot_true_quote_curves(
    artifacts.oracle_models, z_grid,
    title="True marginal fill probability and adverse selection by regime",
)

Fill probability rises with aggressiveness in every regime (left). Adverse
selection A(z) is positive everywhere and largest for *passive* quotes in the
*stressed* regime (right): the only clients who lift a wide, passive quote are
the ones who know something. The post-win value therefore differs from the
unconditional value V0 at every quote, which is exactly what a plain
responder ignores.

# Part III — Learn the static RFQ components

The static components (V0, fill probability, adverse selection) are fitted on
the bond-level synthetic RFQ market from the first notebook, with a strict
chronological split. This part is deliberately compact — the first notebook
treats it in depth.

## Section 5 — Learn V0

In [ ]:
synthetic_config = SyntheticConfig()
market = make_synthetic_rfqs(config=synthetic_config, random_state=SEED, include_latent=True)
framework = fit_framework(market)
train_obs = observable_view(framework.train_df)
test_obs = observable_view(framework.test_df)
value_eval = evaluate_value_models(train_obs, test_obs)
print(value_eval["metrics_table"])

In [ ]:
fig, ax = plots.plot_value_model_comparison(value_eval["metrics_by_model"])
v0_forecast = value_eval["forecasts"]["regularized"]
fig, ax = plots.plot_value_prediction_calibration(
    v0_forecast - test_obs["cp_plus"],
    test_obs["y5"] - test_obs["cp_plus"],
)

The regularized pooled V0 beats both CP+ and the raw internal mid on held-out
MAE, and its predicted deviations from CP+ are well calibrated against realized
deviations. This measures *future-value prediction*, not RFQ profitability.

## Section 6 — Learn fill probability

In [ ]:
fill_eval = evaluate_fill_model(train_obs, test_obs)
fig, ax = plots.plot_fill_calibration(
    fill_eval["calibration_curve"], fill_eval["brier_score"], fill_eval["log_loss"]
)
fill_curve = counterfactual_fill_curve(framework.models.fill_model, test_obs)
empirical = win_rate_by_aggressiveness_bucket(test_obs)
fig, ax = plots.plot_fill_probability_by_aggressiveness(fill_curve, empirical)

The fitted p(q, X) is calibrated on held-out data and reproduces the
monotone aggressiveness-to-fill relationship, so the optimizer can trust its
counterfactual "what if I quoted tighter" queries within the trained support.

## Section 7 — Learn post-win value

On fills, the selection target is `D = side_sign x (V0_oof - Y5)`; the model
learns `A(q, X) = E[D | win, q, X]` and the responder uses
`m(q, X) = V0 - side_sign x A(q, X)`.

In [ ]:
train_fills = train_obs.loc[train_obs["won"] & train_obs["v0_oof"].notna()]
test_fills = test_obs.loc[test_obs["won"] & test_obs["v0_oof"].notna()]
selection_eval = evaluate_selection_model(train_fills, test_fills)
print(format_selection_metrics(selection_eval))

In [ ]:
oracle_context = build_oracle_context(market, synthetic_config)
latent_test_fills = framework.test_df.loc[
    framework.test_df["won"] & framework.test_df["v0_oof"].notna()
]
oracle_sample = latent_test_fills.iloc[:400]
oracle_sel = oracle_selection(oracle_sample, oracle_sample["quote"], oracle_context, n_draws=1000)
predicted_on_sample = predict_selection(
    framework.models.selection_model, observable_view(oracle_sample)
)
fig, ax = plots.plot_predicted_vs_oracle_selection(predicted_on_sample, oracle_sel)

The fitted selection model tracks the oracle's true conditional selection
computed from the hidden data-generating process — evidence that pooled,
regularized estimation recovers the counterfactual quantity the responder
needs, without ever touching a latent column.

# Part IV — The static responder

## Section 8 — Plain versus edge-consistent response

One dealer-buy and one dealer-sell RFQ, priced by the plain CP+ responder,
the plain V0 responder, and the edge-consistent responder on the identical
candidate grid, with the oracle's expected objective attached for reference.

In [ ]:
optimizer_config = OptimizerConfig()
buy_candidates = framework.test_df.loc[
    (framework.test_df["side"] == "dealer_buy") & framework.test_df["v0_oof"].notna()
]
rfq_buy = buy_candidates.iloc[[4]]
scored_buy = score_rfq(framework, rfq_buy, optimizer_config)
grid_columns = [
    "quote", "aggressiveness", "p_win", "selection_points",
    "post_win_value_edge_consistent", "apparent_edge_cents",
    "edge_cents_edge_consistent", "cost_cents", "inventory_value_cents",
    "expected_value_cents_edge_consistent", "in_support",
]
scored_buy["grid"][grid_columns].round(2)

In [ ]:
oracle_grid_buy = oracle_optimal_quote(rfq_buy, oracle_context, optimizer_config)
fig, axes = plots.plot_quote_surface(
    scored_buy["grid"], scored_buy["comparison"], oracle_grid_buy, side_label="(dealer buy)"
)

In [ ]:
sell_candidates = framework.test_df.loc[
    (framework.test_df["side"] == "dealer_sell") & framework.test_df["v0_oof"].notna()
]
rfq_sell = sell_candidates.iloc[[0]]
scored_sell = score_rfq(framework, rfq_sell, optimizer_config)
oracle_grid_sell = oracle_optimal_quote(rfq_sell, oracle_context, optimizer_config)
fig, axes = plots.plot_quote_surface(
    scored_sell["grid"], scored_sell["comparison"], oracle_grid_sell, side_label="(dealer sell)"
)

In [ ]:
comparison_buy = append_oracle_objective(
    scored_buy["comparison"], rfq_buy, oracle_context, optimizer_config
)
comparison_columns = [
    "responder", "accepted", "quote", "aggressiveness", "p_win", "selection_points",
    "conditional_edge_cents", "expected_value_cents", "oracle_expected_objective_cents",
]
comparison_buy[comparison_columns].round(3)

The selected quotes differ for one reason: the plain responders maximize
`p x apparent edge`, which keeps improving as the quote gets more aggressive,
while the edge-consistent responder knows that each extra tick of
aggressiveness buys fills *from better-informed counterparties*, so its
conditional edge rolls over earlier and it quotes more conservatively (or
declines). The signs mirror correctly between the dealer-buy and dealer-sell
panels.

# Part V — Dynamic control and the jump-HJB

## Section 9 — Why the static responder is incomplete

The static responder prices each RFQ in isolation. It does not know:

* the current inventory trajectory,
* the target inventory and deadline,
* the value of future RFQ opportunities,
* future active execution costs.

The dynamic layer replaces the static per-trade inventory adjustment with the
**continuation-value difference of the fill**:

$$\Delta V_{fill} = V\big(t,\; I + \sigma n,\; r\big) - V\big(t,\; I,\; r\big),$$

evaluated at the post-RFQ stage of the same step. The identical RFQ can be
worthless at flat inventory and precious when it cuts an 8-unit shortfall:

In [ ]:
liq_config = liquidation_episode()
liq_solutions = solve_episode_policies(artifacts, liq_config)
liq_solution = liq_solutions["DynamicExecution"]
helpful_event = RFQEvent(
    event_id=0, time_index=20, side="dealer_sell", side_sign=-1, size=2,
    client_tier="professional", liquidity_score=0.55, market_width=0.12,
    regime=MarketRegime.NORMAL, cp_plus=100.0,
    hidden_client_signal=0.0, hidden_future_residual=0.0,
)
rfq_decision_across_inventories(
    artifacts.market_config, artifacts.fitted_models, liq_solution,
    inventories=(0, 2, 4, 8), time_index=20, event=helpful_event,
).round(2)

At flat inventory the dealer-sell RFQ is declined — its continuation delta is
*negative* (it would create a short). At +4 and +8 the same RFQ carries a
large positive continuation delta, the quote gets aggressive, and the fill
becomes passive execution. No mode switch was coded anywhere; only
`Delta V_fill` changed.

## Section 10 — The Bellman recursion

The solver uses exact backward induction with the same event ordering as the
forward simulator: (1) observe state, (2) observe RFQ, (3) quote or decline,
(4) apply the fill transition, (5) choose active execution, (6) pay the
running penalty, (7) regime and time transition.

**Terminal condition**
$$V_T(I, r) = -\eta\,(I - I^*)^2$$

**RFQ jump increment** (for an RFQ of side sign sigma and size n; U is the
post-RFQ-stage continuation of the same step)
$$\mathrm{RFQIncrement}(q) = p(q, X)\,\big[\, r_{rfq}(q, X, n) + U(I + \sigma n,\, r) - U(I,\, r) \,\big]$$
with response only if the best increment is strictly positive.

**Active execution operator**
$$U(I, r) = \max_u \Big[ -C_{active}(u, X) - \phi\,(I + u - I^*)^2 + \mathbb{E}\big[ V_{k+1}(I + u,\, r') \,\big|\, r \big] \Big]$$

**This notebook solves a finite-state, finite-horizon Bellman problem. It is
a numerical approximation to a jump-HJB, not an exact continuous-time PDE
solution.** The residual check below re-evaluates the recursion at the solved
values:

In [ ]:
residual_stats = bellman_residual(liq_solution)
print(f"max |residual|:  {residual_stats['max_abs_residual_cents']:.3e} cents")
print(f"mean |residual|: {residual_stats['mean_abs_residual_cents']:.3e} cents")
print(f"states above tolerance ({residual_stats['tolerance_cents']:.0e} cents): "
      f"{residual_stats['n_states_violating_tolerance']}")
fig, ax = control_plots.plot_bellman_residual(
    liq_solution, title="Bellman residual by time step (liquidation problem)"
)

Backward induction is exact on this grid, so the residual sits at floating-
point noise, far below the 1e-6 cent tolerance, at every step and state.

## Section 11 — Visualize the value function

In [ ]:
fig, ax = control_plots.plot_value_function(
    liq_solution, regime_index=MarketRegime.NORMAL.value,
    steps=(0, 20, 35, 40),
    title="Liquidation value V_k(I, NORMAL): the terminal penalty pulls inventory to 0",
)
fig, ax = control_plots.plot_inventory_shadow_value(
    liq_solution, regime_index=MarketRegime.NORMAL.value, steps=(0, 20, 35, 39),
    title="Inventory shadow value dV/dI (liquidation, NORMAL regime)",
)

Early in the episode (k = 0) the value surface is gentle: there is time to
work the position through RFQs. Only at the horizon itself does the surface
become the terminal parabola $-\eta (I - I^*)^2$ (the k = 40 curve). The
shadow value panel prices one extra long unit at roughly -25 cents through
most of the episode — note this is *not* the terminal slope: as long as steps
remain, active execution caps the marginal cost of inventory at its expected
unwind cost (half-spread, impact, and fee). The shadow value steepens near
the deadline exactly when that unwind capacity runs out. This is the number
an RFQ quote should internalize, and it is what `Delta V_fill` feeds into the
RFQ jump operator.

In [ ]:
fig, ax = control_plots.plot_value_function(
    liq_solution, regime_index=MarketRegime.STRESSED_ILLIQUID.value,
    steps=(0, 20, 35, 40),
    title="Liquidation value V_k(I, STRESSED): scarcer RFQs and costlier unwinds",
)

The stressed-regime surface is uniformly lower and more curved away from the
target: RFQs arrive less often, active execution costs more, and adverse
selection is stronger, so carrying the same shortfall is more expensive.

In [ ]:
fig, ax = control_plots.plot_active_execution_policy_heatmap(
    liq_solution, regime_index=MarketRegime.NORMAL.value,
    title="Optimal active execution u(k, I) — liquidation, NORMAL regime",
)
fig, ax = control_plots.plot_quote_policy_heatmap(
    liq_solution, regime_index=MarketRegime.NORMAL.value, side_sign=-1, size=1,
    title="Optimal quote aggressiveness for dealer-sell RFQs — liquidation, NORMAL",
)

The active heatmap shows the classic execution wedge: wait while the clock is
long and inventory is near target, sell harder (u = -1, then -2) as the
deadline approaches with inventory still long. The quote heatmap shows the
same urgency through the RFQ channel: dealer-sell RFQs (which cut a long) are
quoted more and more aggressively above the target line and declined (blank)
below it, where a fill would push inventory the wrong way.

# Part VI — Market-making demonstration

## Section 12 — The dynamic market maker

In [ ]:
mm_solutions = solve_episode_policies(artifacts, mm_config)
mm_policies = make_policies(artifacts, mm_config, mm_solutions)
mm_path = simulate_exogenous_path(artifacts.market_config, mm_config, random_state=10)
mm_result = simulate_episode(
    mm_policies["DynamicMarketMaker"], mm_config, artifacts.market_config, mm_path
)
fig, ax = control_plots.plot_event_timeline(
    mm_result.log, title="Market-making episode: RFQ stream, fills, and active trades"
)
fig, ax = control_plots.plot_inventory_path(
    mm_result.log, title="Market-making episode: inventory oscillates around zero"
)

In [ ]:
fig, ax = control_plots.plot_mode_timeline(
    mm_result.log, title="Market-making episode: economic mode per step"
)
fig, ax = control_plots.plot_cumulative_reward(
    mm_result.log, title="Market-making episode: cumulative simulated control reward"
)
components = reconcile_episode_rewards(mm_result.log)
print(pd.Series(components).round(2).to_string())

The market maker is *selective*: in this calibration adverse selection eats
most of the quoted width, so it quotes only when the selection-adjusted edge
clears costs plus the (small) inventory continuation cost, and it declines the
rest. Fills nudge inventory off zero and the running penalty pulls it back.
The reconciliation table confirms the episode's reward components sum exactly
to the cumulative total.

**The same RFQ, quoted differently because inventory differs.** The table
below prices one identical client-buy RFQ (dealer sells 1 unit) at three
inventory levels under the market-making value function:

In [ ]:
mm_solution = mm_solutions["DynamicMarketMaker"]
mm_event = RFQEvent(
    event_id=0, time_index=30, side="dealer_sell", side_sign=-1, size=1,
    client_tier="professional", liquidity_score=0.55, market_width=0.12,
    regime=MarketRegime.NORMAL, cp_plus=100.0,
    hidden_client_signal=0.0, hidden_future_residual=0.0,
)
rfq_decision_across_inventories(
    artifacts.market_config, artifacts.fitted_models, mm_solution,
    inventories=(-6, 0, 6), time_index=30, event=mm_event,
).round(2)

The identical RFQ gets three different answers. Short 6 units, selling one
more unit is firmly declined (continuation delta about -25 cents). Flat, the
standalone selection-adjusted edge is positive (+8.6 cents) but *still not
enough*: going one unit short costs slightly more in expected running penalty
and unwind than the trade earns, so the market maker declines by a whisker.
Long 6 units, the same fill now cuts inventory risk, the continuation delta
flips to +23 cents, and the RFQ is quoted aggressively. Defensive and
opportunistic behavior fall out of one formula.

**Declining despite positive apparent edge.** In the stressed regime a wide
market makes passive quotes *look* attractive (apparent edge = -z x width is
large and positive for passive z), yet the controller declines:

In [ ]:
stressed_event = RFQEvent(
    event_id=0, time_index=30, side="dealer_buy", side_sign=1, size=2,
    client_tier="informed", liquidity_score=0.20, market_width=0.25,
    regime=MarketRegime.STRESSED_ILLIQUID, cp_plus=100.0,
    hidden_client_signal=0.0, hidden_future_residual=0.0,
)
stressed_curves = rfq_increment_curves(
    artifacts.market_config, artifacts.fitted_models, mm_solution,
    inventory=8, time_index=30, event=stressed_event,
)
apparent_edge_cents = (
    -stressed_curves["aggressiveness"] * stressed_event.market_width * 100.0 * stressed_event.size
)
best_row = stressed_curves.loc[stressed_curves["rfq_increment_cents"].idxmax()]
print(f"Best apparent edge on the grid:      {float(apparent_edge_cents.max()):+.1f} cents")
print(f"Best selection-adjusted standalone:  {float(stressed_curves['standalone_reward_cents'].max()):+.1f} cents")
print(f"Continuation delta of a fill:        {float(best_row['continuation_delta_cents']):+.1f} cents")
print(f"Best RFQ increment:                  {float(best_row['rfq_increment_cents']):+.2f} cents -> "
      f"{'QUOTE' if best_row['rfq_increment_cents'] > 0 else 'DECLINE'}")
fig, axes = control_plots.plot_quote_decision_at_event(
    aggressiveness_grid=stressed_curves["aggressiveness"].to_numpy(),
    fill_probability=stressed_curves["p_win"].to_numpy(),
    trade_reward_cents=stressed_curves["standalone_reward_cents"].to_numpy(),
    increments_cents=stressed_curves["rfq_increment_cents"].to_numpy(),
    chosen_aggressiveness=None,
    title="Stressed dealer-buy RFQ at +8 inventory: positive apparent edge, decline anyway",
)

A passive quote at z = -1.5 shows roughly +75 cents of *apparent* edge on this
2-unit RFQ. But stressed-regime selection removes most of it, and buying two
more units at +8 inventory carries a negative continuation delta — so every
candidate's RFQ increment is at or below the decline value of zero. The plain
responder would have quoted here; the dynamic controller correctly does not.

# Part VII — Position-execution demonstration

## Section 13 — Liquidate a long position

Initial inventory +8, target 0, 40 steps, terminal penalty 60 cents per
squared unit. Client-buy RFQs (dealer sells) help; client-sell RFQs would
enlarge the position.

In [ ]:
liq_policies = make_policies(artifacts, liq_config, liq_solutions)
liq_path = simulate_exogenous_path(artifacts.market_config, liq_config, random_state=10)
liq_result = simulate_episode(
    liq_policies["DynamicExecution"], liq_config, artifacts.market_config, liq_path
)
fig, ax = control_plots.plot_inventory_path(
    liq_result.log, title="Liquidation episode: inventory works down to the target"
)
fig, ax = control_plots.plot_event_timeline(
    liq_result.log, title="Liquidation episode: RFQs, fills, and active sales"
)

In [ ]:
fig, ax = control_plots.plot_target_shortfall(
    liq_result.log, title="Liquidation episode: remaining target shortfall"
)
fig, ax = control_plots.plot_mode_timeline(
    liq_result.log, title="Liquidation episode: economic mode per step"
)
fig, ax = control_plots.plot_cumulative_reward(
    liq_result.log, title="Liquidation episode: cumulative simulated control reward"
)

In [ ]:
liq_summary = summarize_episode_log(liq_result.log, liq_config)
print(pd.Series({
    "terminal shortfall (units)": liq_summary["terminal_target_shortfall"],
    "completion (%)": liq_summary["target_completion_pct"],
    "passively internalized (units)": liq_summary["passive_internalized_units"],
    "active volume (units)": liq_summary["active_volume_units"],
    "share moved via RFQs": liq_summary["proportion_via_rfqs"],
    "active execution cost (c)": liq_summary["active_execution_cost_cents"],
    "adverse selection paid (c)": liq_summary["adverse_selection_cents"],
    "total reward (c)": liq_summary["total_objective_cents"],
}).round(2).to_string())

The pattern the theory predicts is visible: early on the controller *waits*
for helpful client-buy RFQs and internalizes them (green passive-execution
bands in the mode timeline); dealer-buy RFQs are declined or quoted only
defensively; as the deadline approaches with shortfall left, it switches to
paid active sales (orange). The internalized share is the cheap part of the
liquidation — every unit sold to a client through an RFQ saves the active
half-spread and impact.

## Section 14 — Build a long position

Same machinery, reversed target: initial inventory 0, target +8. Now
client-**sell** RFQs (dealer buys) are the helpful side, and client-buy RFQs
move inventory away from the target.

In [ ]:
acq_config = acquisition_episode()
acq_solutions = solve_episode_policies(artifacts, acq_config)
acq_policies = make_policies(artifacts, acq_config, acq_solutions)
acq_path = simulate_exogenous_path(artifacts.market_config, acq_config, random_state=7)
acq_result = simulate_episode(
    acq_policies["DynamicExecution"], acq_config, artifacts.market_config, acq_path
)
fig, ax = control_plots.plot_inventory_path(
    acq_result.log, title="Acquisition episode: inventory builds up to +8"
)
fig, ax = control_plots.plot_mode_timeline(
    acq_result.log, title="Acquisition episode: economic mode per step"
)

In [ ]:
acq_event = RFQEvent(
    event_id=0, time_index=20, side="dealer_buy", side_sign=1, size=2,
    client_tier="professional", liquidity_score=0.55, market_width=0.12,
    regime=MarketRegime.NORMAL, cp_plus=100.0,
    hidden_client_signal=0.0, hidden_future_residual=0.0,
)
acq_solution = acq_solutions["DynamicExecution"]
rfq_decision_across_inventories(
    artifacts.market_config, artifacts.fitted_models, acq_solution,
    inventories=(0, 4, 8), time_index=20, event=acq_event,
).round(2)

The helpful and harmful sides have flipped exactly as the inventory algebra
requires: the dealer-buy RFQ now carries a large positive continuation delta
at inventory 0 (it builds the long) and is declined at +8 (target reached).

# Part VIII — Compare all policies

## Section 15 — Aggregate comparison

500 identical seeded episodes per configuration; every policy sees the same
exogenous regime, price, RFQ, and fill-uniform paths (common random numbers).
The total is **simulated control reward**, never real trading PnL.

In [ ]:
evaluation = evaluate_control_policies(
    policy_names=POLICY_ORDER,
    episode_configs=[market_making_episode(), liquidation_episode(), acquisition_episode()],
    artifacts=artifacts,
    n_episodes=500,
    random_state=2024,
    bootstrap_samples=500,
)
print("=== Market making (mean per episode, 500 episodes) ===")
policy_comparison_table(evaluation, "market_making")

In [ ]:
print("=== Liquidation (mean per episode, 500 episodes) ===")
policy_comparison_table(evaluation, "liquidation")

In [ ]:
print("=== Acquisition (mean per episode, 500 episodes) ===")
policy_comparison_table(evaluation, "acquisition")

In [ ]:
format_paired_differences(evaluation)

Reading the paired block-bootstrap intervals (episode blocks, 95%):

* In **execution episodes** the dynamic controllers complete the target
  essentially always, while both myopic responders leave most of the position
  and absorb terminal penalties in the thousands of cents. The differences
  are large and their intervals exclude zero.
* In **market making** the ordering is Plain < EdgeConsistentMyopic <
  Dynamic < Oracle. The plain responder's loss is the price of ignoring
  adverse selection; the dynamic policies avoid negative-value trades and
  keep inventory near zero.
* The `DynamicMarketMaker` fails the *acquisition* episode by construction —
  it plans for target zero — which is precisely the point of separating the
  two dynamic policies.
* The oracle rows are an upper bound from knowing the hidden client signal;
  they are simulation diagnostics, not an attainable strategy.

In [ ]:
fig, ax = control_plots.plot_policy_comparison(
    evaluation.policy_metrics, "liquidation",
    metric="total_objective_cents",
    ylabel="mean simulated control reward (cents / episode)",
    title="Liquidation: total simulated control reward by policy",
)
fig, ax = control_plots.plot_reward_decomposition(
    evaluation.episode_summaries, "liquidation",
    title="Liquidation: reward decomposition by policy (mean per episode)",
)

In [ ]:
fig, ax = control_plots.plot_completion_cost_frontier(
    evaluation.episode_summaries, "liquidation",
    title="Liquidation: completion versus execution cost",
)
fig, ax = control_plots.plot_internalization_fraction(
    evaluation.policy_metrics, "liquidation",
    title="Liquidation: share of target-directed volume internalized via RFQs",
)
fig, ax = control_plots.plot_regime_performance(
    evaluation.regime_metrics, metric="response_rate",
    ylabel="RFQ response rate",
    title="Response rate by market regime (all episodes pooled)",
)

The decomposition shows *where* the myopic policies lose: not on the RFQs they
win (their clean edge is comparable) but on the terminal shortfall they never
address. The frontier makes the trade explicit — the dynamic controllers buy
~100% completion for roughly a hundred cents of execution cost, versus
thousands of cents of penalty avoided. The regime panel shows every policy
becoming more selective as markets become stressed, with the dynamic policies
cutting response rates the hardest where selection is worst.

## Section 16 — Where does the improvement come from?

Ablations on the liquidation episode, all on identical paths:

In [ ]:
ablation = run_ablation_study(
    artifacts, liquidation_episode(), n_episodes=200, random_state=3,
)
ablation.round(1)

In [ ]:
fig, ax = control_plots.plot_ablation_totals(
    ablation, title="Liquidation ablations: mean total reward per variant (200 episodes)"
)

The attribution is clean:

* removing the **continuation value** (myopic) or the **selection adjustment**
  (plain) is catastrophic — the position never gets liquidated;
* removing **active execution** still completes ~90% via RFQs but pays
  terminal penalties when flow dries up;
* removing **RFQ internalization** completes via active trades alone, at a
  visibly higher execution cost than the full controller;
* upgrading either fitted ingredient to its **oracle** counterpart adds only a
  few cents — the fitted models are close to the truth in this simulation —
  and the full oracle bounds the achievable reward.

# Part IX — Robustness

## Section 17 — Sensitivity analysis

The liquidation comparison is repeated across 16 scenario variants: inventory
penalty, terminal penalty, RFQ cost (5 / 7.5 / 10 cents), adverse-selection
strength, active market impact, deadline length, and RFQ arrival intensity.
Scenarios that change the data-generating process refit the control models on
the new market.

In [ ]:
sensitivity = run_control_sensitivity(
    base_episode_config=liquidation_episode(),
    base_artifacts=artifacts,
    policy_names=("EdgeConsistentMyopic", "DynamicExecution", "OracleDynamic"),
    n_episodes=40,
    random_state=5,
)
print("Mean total simulated control reward (cents / episode):")
print(
    sensitivity.pivot_table(index="scenario", columns="policy",
                            values="total_objective_cents", sort=False)
    .round(0).to_string()
)
print()
print("Decline rate:")
print(
    sensitivity.pivot_table(index="scenario", columns="policy",
                            values="decline_rate", sort=False)
    .round(2).to_string()
)

In [ ]:
fig, ax = control_plots.plot_sensitivity_heatmap(
    sensitivity, metric="target_completion_pct",
    title="Sensitivity: target completion (%) by scenario and policy",
)
fig, ax = control_plots.plot_sensitivity_heatmap(
    sensitivity, metric="active_volume_units",
    title="Sensitivity: active execution volume by scenario and policy",
)

The qualitative conclusions are stable everywhere: the dynamic controller
completes the target in every scenario, substituting between the RFQ and
active channels as their relative prices move — more active volume when
arrivals are scarce or the deadline is short, more internalization when
impact is high or flow is rich. The myopic responder's completion is poor in
every variant; no cost assumption rescues it.

The mode structure of the solved policy summarizes the whole controller in
one picture:

In [ ]:
fig, ax = control_plots.plot_mode_map(
    liq_solution, regime_index=MarketRegime.NORMAL.value, side_sign=-1, size=1,
    title="Mode map for dealer-sell RFQs (liquidation, NORMAL): the emergent regions",
)

Above the target the helpful dealer-sell RFQ is quoted as passive execution
(green); at and below the target the same RFQ is declined or handled as
market making; active execution (orange) appears only where the deadline
would otherwise be missed. These regions were never labelled by hand — they
are read off the optimal actions after the fact.

## Section 18 — Sparse and cold-start behavior

The static value model that feeds the responder pools information across
bonds and issuers, so its behavior degrades gracefully as history thins out:

In [ ]:
cold_start = cold_start_comparison(framework, optimizer_config)
cold_start.round(3)

An actively traded bond gets a bond-specific V0; a sparse bond leans on its
issuer; an unseen bond from a known issuer inherits the issuer's level; an
unseen issuer falls back toward CP+ with near-zero predicted deviation and a
conservative selection estimate. The same conservatism protects the dynamic
controller, whose planning tables are regime-level aggregates rather than
bond-specific estimates.

# Part X — Conclusions

## Section 19 — What the framework establishes

* **The quote is a two-sided control.** It moves the fill probability *and*
  the post-win value, because winning is informative. Pricing only the first
  effect (the plain responder) systematically buys adversely selected trades.
* **The inventory continuation value decides what an RFQ is.** The same
  quote formula produces market making when `Delta V_fill ~ 0`, passive
  execution when the fill cuts the target shortfall, and defensive quoting or
  declines when it adds to it.
* **Active execution appears endogenously** when RFQ flow is insufficient —
  late in the episode, in stressed regimes, or when arrivals are scarce — and
  disappears when internalization is cheaper.
* **Urgency is priced, not scripted.** The shadow value of inventory steepens
  as the deadline approaches, which mechanically tightens helpful quotes and
  raises active volume.
* **The mode labels are outputs.** Market making versus execution emerges
  from the state and the value function, not from a hard-coded switch.

The decision structure the static responder was missing, in one diagram:

```text
Standalone positive RFQ edge                    ->  MARKET MAKING
RFQ moves inventory toward target               ->  PASSIVE EXECUTION
RFQ moves inventory away from target            ->  DEFENSIVE QUOTING OR DECLINE
Deadline approaches and RFQs are insufficient   ->  ACTIVE EXECUTION
```

## Section 20 — Limitations

* **Synthetic results are not live profitability evidence.** Every number is
  a property of the simulator's assumptions.
* **The control problem is discretized**: integer inventory, a finite quote
  grid, a five-point active grid, and three regimes. The solver is a discrete
  Bellman approximation to a jump-HJB, not an exact continuous-time solution.
* **Real RFQ arrival intensities and counterfactual fill behavior must be
  estimated empirically**; here they are known simulator inputs, which
  flatters every fitted model.
* **The t+5 clean-value clock and the execution clock are kept distinct here
  and must remain so in practice** — conflating them double-counts alpha.
* **Impact, hedging, and cross-bond substitution are simplified**: active
  impact is temporary and quadratic, there is no hedge instrument, and each
  episode trades a single bond.
* **Costs and penalties are calibration choices**; the sensitivity section
  shows the conclusions that survive them and the magnitudes that do not.